In [0]:
from pyspark.sql.functions import col, when, regexp_replace

# Source
incoming_path = (
    "/Volumes/workspace/bronze/scada_raw_files/"
    "autoloader_demo/incoming"
)

# Auto Loader state
schema_path = (
    "/Volumes/workspace/bronze/scada_raw_files/"
    "autoloader_demo/schema"
)

bronze_checkpoint = (
    "/Volumes/workspace/bronze/scada_raw_files/"
    "autoloader_demo/checkpoint"
)

silver_checkpoint = (
    "/Volumes/workspace/bronze/scada_raw_files/"
    "autoloader_demo/silver_checkpoint"
)

gold_checkpoint = (
    "/Volumes/workspace/bronze/scada_raw_files/"
    "autoloader_demo/gold_checkpoint"
)

# Unity Catalog tables
bronze_table = "workspace.bronze.scada_sensor_stream_raw"
silver_table = "workspace.silver.scada_sensor_stream_clean"
gold_table = "workspace.gold.sensor_stream_summary"

print("Pipeline configuration loaded")

Pipeline configuration loaded


In [0]:
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("delimiter", ";")
    .load(incoming_path)
)

bronze_query = (
    bronze_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", bronze_checkpoint)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

bronze_query.awaitTermination()

print("Bronze ingestion completed")

Bronze ingestion completed


In [0]:
silver_stream_df = (
    spark.readStream
    .table(bronze_table)
    .dropna(
        how="all",
        subset=["id", "value", "unit", "timestamp"]
    )
    .withColumn("id", col("id").cast("int"))
    .withColumn(
        "value",
        when(
            col("value").rlike(r"^[0-9]+\.[0-9]+\.[0-9]+$"),
            regexp_replace(
                col("value"),
                r"^([0-9]+)\.([0-9]+)\.([0-9]+)$",
                "$1$2.$3"
            )
        )
        .otherwise(col("value"))
        .cast("double")
    )
    .withColumn(
        "timestamp",
        col("timestamp").cast("timestamp")
    )
    .select("id", "value", "unit", "timestamp")
)

silver_query = (
    silver_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .toTable(silver_table)
)

silver_query.awaitTermination()

print("Silver transformation completed")

Silver transformation completed


In [0]:
from pyspark.sql.functions import count, avg, min, max

gold_stream_df = (
    spark.readStream
    .table(silver_table)
    .groupBy("id", "unit")
    .agg(
        count("*").alias("reading_count"),
        avg("value").alias("avg_value"),
        min("value").alias("min_value"),
        max("value").alias("max_value")
    )
)

gold_query = (
    gold_stream_df.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", gold_checkpoint)
    .trigger(availableNow=True)
    .toTable(gold_table)
)

gold_query.awaitTermination()

print("Gold aggregation completed")

Gold aggregation completed


In [0]:
print(
    "Bronze:",
    spark.table("workspace.bronze.scada_sensor_stream_raw").count()
)

print(
    "Silver:",
    spark.table("workspace.silver.scada_sensor_stream_clean").count()
)

print(
    "Gold:",
    spark.table("workspace.gold.sensor_stream_summary").count()
)

Bronze: 200000
Silver: 182866
Gold: 32


In [0]:
print(
    "Bronze:",
    spark.table("workspace.bronze.scada_sensor_stream_raw").count()
)

print(
    "Silver:",
    spark.table("workspace.silver.scada_sensor_stream_clean").count()
)

print(
    "Gold:",
    spark.table("workspace.gold.sensor_stream_summary").count()
)

Bronze: 300000
Silver: 282866
Gold: 32
